# Linear Regression

> 📘 **Python Mastery** · Module 13 — Machine Learning · Lesson 4/7

The first model everyone meets, and still one of the most used in industry:
draw the straight line that best explains how an input drives an outcome.

## 🎯 Learning Objectives

- Generate realistic synthetic data with NumPy's seeded random generator.
- Fit `LinearRegression` and read the learned equation from `coef_` and `intercept_`.
- Visualise raw data and the fitted line on the same axes.
- Predict prices for unseen house sizes.
- Compute MAE, MSE, RMSE, and R² by hand, then verify with sklearn.
- Run residual diagnostics to check whether a line was the right shape at all.
- Extend to multiple features and preview polynomial regression for curves.

## 1. The Idea: Draw the Best Line

You already do this mentally. A friend shows you a 1,200 sqft flat and asks what it's
worth — you slide along your internal *"price per sqft"* experience and answer.
Linear regression does exactly that, but writes the line down precisely:

$$\hat{y} = w \cdot x + b$$

- $w$ (**weight/slope**) — how many lakh Taka each extra sqft adds,
- $b$ (**bias/intercept**) — the baseline price at zero sqft,
- $\hat{y}$ ("y-hat") — the *predicted* value, as opposed to the true $y$.

"Training" means choosing the one line out of infinitely many that keeps prediction
errors smallest.

**Syntax:**
```python
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X, y)                 # X must be 2-D: (n_samples, n_features)
model.predict(X_new)            # same shape convention
model.coef_[0], model.intercept_   # that's w and b
```

## 2. Crafting the Data

We simulate the Dhaka housing market so ground truth is known: a base of 25 lakh
plus about 0.085 lakh per sqft, plus life's noise (renovations, view, negotiation).

**Syntax:**
```python
import numpy as np

rng = np.random.default_rng(42)          # seeded => identical data every run
size_sqft = rng.uniform(700, 3000, n)    # feature
noise     = rng.normal(0, 18, n)         # real-world messiness
y         = 25 + 0.085 * size_sqft + noise
```

> 🔍 **Under the Hood:** ordinary least squares needs **no coin flips at all**.
> `.fit()` solves for the coefficients in closed form (via singular value
> decomposition under the hood), projecting `y` onto the column space of `X`.
> That is why `LinearRegression` has no `random_state` — run it twice and you get
> bit-identical weights. Contrast that with K-Means or neural nets, which start
> from random guesses and need seeds.

In [ ]:
# Build a synthetic housing dataset (deterministic)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)
n = 80

size_sqft  = np.round(rng.uniform(700, 3000, n))         # apartment size
price_lakh = 25 + 0.085 * size_sqft + rng.normal(0, 18, n)  # truth + noise

df = pd.DataFrame({"size_sqft": size_sqft, "price_lakh": price_lakh.round(1)})
print(df.head())
print("\nCorrelation(size, price):", df["size_sqft"].corr(df["price_lakh"]).round(3))

plt.figure(figsize=(7, 4))
plt.scatter(df["size_sqft"], df["price_lakh"], s=28, alpha=0.75, color="#1f77b4")
plt.title("Dhaka flats: size vs price (synthetic)")
plt.xlabel("Size (sqft)")
plt.ylabel("Price (lakh Taka)")
plt.grid(alpha=0.3)
plt.show()

## 3. Fitting the Model

One call. Then interrogate it: `coef_` is the slope, `intercept_` is where the line
crosses zero. Together they ARE the model.

In [ ]:
# Fit and read off the learned equation
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
import pandas as pd
import numpy as np

X = df[["size_sqft"]]                      # 2-D DataFrame: rows x ONE feature
y = df["price_lakh"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

model = LinearRegression().fit(X_train, y_train)

w = model.coef_[0]
b = model.intercept_
print(f"learned equation: price_hat = {w:.4f} * size + {b:.2f}")
print(f"truth we planted : price_hat = 0.0850 * size + 25.00")
print("\nThe model recovered the hidden pattern from noisy data alone.")

## 4. Overlaying the Line

A picture beats a coefficient table: scatter the data, then sweep a straight line
through the fitted equation across the full size range.

In [ ]:
# Scatter + fitted line on one canvas
import matplotlib.pyplot as plt
import numpy as np

sizes_grid = np.linspace(X_train["size_sqft"].min(), X_train["size_sqft"].max(), 100)
grid_df = pd.DataFrame({"size_sqft": sizes_grid})
line = model.predict(grid_df)

plt.figure(figsize=(7, 4))
plt.scatter(X_train["size_sqft"], y_train, s=28, alpha=0.75,
            color="#1f77b4", label="training flats")
plt.plot(sizes_grid, line, color="#d62728", linewidth=2, label="fitted line")
plt.title("Learned relationship: price grows ~{:.1f} lakh per 1000 sqft".format(w * 1000))
plt.xlabel("Size (sqft)")
plt.ylabel("Price (lakh Taka)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 5. Predicting New Flats

Inference time: feed fresh sizes, get prices. Note the double brackets —
sklearn wants a 2-D input even for one row (`[[1600]]`, or a DataFrame like ours).

In [ ]:
# Price three flats the model never saw
import pandas as pd

new_flats = pd.DataFrame({"size_sqft": [950, 1600, 2600]})
new_flats["predicted_price_lakh"] = model.predict(new_flats).round(1)
print(new_flats.to_string(index=False))

print("\nEach extra 1000 sqft costs about {:.1f} lakh.".format(w * 1000))
print("Sanity check - a 0 sqft 'flat' would cost {:.1f} lakh (the baseline b).".format(b))

## 6. How Wrong Is It? Error Metrics

Predictions differ from truth by residuals ($y_i - \hat{y}_i$). Four standard ways
to boil all residuals into one number:

| Metric | Formula | Units | Outlier sensitivity | Read it as |
|---|---|---|---|---|
| MAE | $\frac{1}{n}\sum \lvert y_i - \hat{y}_i \rvert$ | lakh Taka | gentle | "average miss" |
| MSE | $\frac{1}{n}\sum (y_i - \hat{y}_i)^2$ | lakh² | brutal | optimisation-friendly loss |
| RMSE | $\sqrt{\text{MSE}}$ | lakh Taka | strong | typical miss, big errors punished |
| R² | $1 - \frac{SS_{res}}{SS_{tot}}$ | none | — | share of variance explained (1 = perfect, 0 = mean-only) |

We compute all four **by hand first** — then let sklearn agree with us.

In [ ]:
# Metrics BY HAND on the held-out test set
import numpy as np

y_pred = model.predict(X_test)
residuals = y_test - y_pred                    # truth minus prediction

mae  = np.abs(residuals).mean()
mse  = (residuals ** 2).mean()
rmse = np.sqrt(mse)
r2   = 1 - (residuals ** 2).sum() / ((y_test - y_test.mean()) ** 2).sum()

print(f"MAE : {mae:.2f} lakh   <- typical miss")
print(f"MSE : {mse:.1f}  lakh^2 (units squared - hard to explain to clients)")
print(f"RMSE: {rmse:.2f} lakh   <- back in real units, outliers punished")
print(f"R^2 : {r2:.3f}        <- {r2:.0%} of price variance explained")

In [ ]:
# sklearn confirms our arithmetic exactly
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print(f"sklearn MAE : {mean_absolute_error(y_test, y_pred):.2f}")
print(f"sklearn MSE : {mean_squared_error(y_test, y_pred):.1f}")
print(f"sklearn RMSE: {mean_squared_error(y_test, y_pred) ** 0.5:.2f}")
print(f"sklearn R^2 : {r2_score(y_test, y_pred):.3f}")

Rule of thumb for talking to stakeholders: quote **RMSE** when large errors hurt
disproportionately (delivery ETAs, dosage estimates), **MAE** when every lakh of
error hurts equally, and **R²** when comparing against a naive baseline. An R² of
0.85 does *not* mean "85% accurate" — it means your model explains 85% of the
spread that a plain average would miss.

## 7. Residual Diagnostics: Was a Line Even Right?

Residuals are the model's confessions. Plot them and they testify:

- **Healthy:** scattered randomly around zero, no shape, similar spread everywhere.
- **Curved pattern:** the truth isn't linear → try polynomial features (below).
- **Funnel shape:** error grows with size → heteroscedasticity; consider log-transforms.

In [ ]:
# Two-panel residual check: pattern hunt + distribution
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(y_pred, residuals, s=26, alpha=0.75, color="#1f77b4")
axes[0].axhline(0, color="#d62728", linewidth=1.5)
axes[0].set_title("Residuals vs predicted")
axes[0].set_xlabel("Predicted price (lakh)")
axes[0].set_ylabel("Residual (lakh)")
axes[0].grid(alpha=0.3)

axes[1].hist(residuals, bins=15, color="#1f77b4", edgecolor="white")
axes[1].axvline(0, color="#d62728", linewidth=1.5)
axes[1].set_title("Distribution of residuals")
axes[1].set_xlabel("Residual (lakh)")

plt.show()
print("No curve, no funnel -> a straight line was a fair assumption here.")

## 8. Multiple Linear Regression

Real prices depend on more than area. Same API — `coef_` simply becomes one weight
per feature, and the equation grows:

$$\hat{y} = w_1 x_1 + w_2 x_2 + b$$

In [ ]:
# Two features: size + number of bedrooms
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

rng = np.random.default_rng(42)
n = 120
size_sqft = np.round(rng.uniform(700, 3000, n))
bedrooms  = rng.integers(2, 6, n)
price     = 15 + 0.075 * size_sqft + 12 * bedrooms + rng.normal(0, 15, n)

X2 = pd.DataFrame({"size_sqft": size_sqft, "bedrooms": bedrooms})
y2 = pd.Series(price, name="price_lakh")

model2 = LinearRegression().fit(X2, y2)
print(f"price_hat = {model2.intercept_:.2f}"
      f" + {model2.coef_[0]:.4f}*size + {model2.coef_[1]:.2f}*bedrooms")

new_flat = pd.DataFrame({"size_sqft": [1400], "bedrooms": [3]})
print(f"a 1400 sqft, 3-bedroom flat -> {model2.predict(new_flat)[0]:.1f} lakh")

## 9. When Can You Trust a Linear Model?

Four assumptions sit underneath every coefficient you report:

- **Linearity** — the effect really is proportional (check: residual plot flat around 0).
- **Independence** — one flat's price doesn't influence another's (beware grouped data).
- **Homoscedasticity** — error spread stays constant across predictions (no funnels).
- **Normality of residuals** — errors roughly bell-shaped (matters for confidence intervals).

Violate linearity badly and the model won't just lose accuracy — it will assert
nonsense slopes with confidence. Diagnose before you deploy.

## 10. Beyond Straight Lines: Polynomial Teaser

When residuals curve, bend the model instead of abandoning it:
`PolynomialFeatures` manufactures extra columns ($x$, $x^2$, $x^3$, …) and the
*fitted line through those columns* becomes a smooth curve through the original ones.

In [ ]:
# Curved data: degree 1 fails, degree 3 fits
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score

rng = np.random.default_rng(7)
x = rng.uniform(-3, 3, 70)
y = 0.6 * x ** 2 - 1.5 * x + 2 + rng.normal(0, 0.8, 70)
X = pd.DataFrame({"x": x})

grid = pd.DataFrame({"x": np.linspace(x.min(), x.max(), 200)})
deg1 = make_pipeline(PolynomialFeatures(degree=1), LinearRegression()).fit(X, y)
deg3 = make_pipeline(PolynomialFeatures(degree=3), LinearRegression()).fit(X, y)

plt.figure(figsize=(7, 4))
plt.scatter(x, y, s=24, alpha=0.7, color="#1f77b4", label="data")
plt.plot(grid["x"], deg1.predict(grid), color="#ff7f0e", linewidth=2, label="degree 1")
plt.plot(grid["x"], deg3.predict(grid), color="#2ca02c", linewidth=2, label="degree 3")
plt.title("Same algorithm, richer features")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.grid(alpha=0.3)
plt.show()

print(f"degree 1 R^2: {deg1.score(X, y):.3f}")
print(f"degree 3 R^2: {deg3.score(X, y):.3f}")
print("(but crank the degree high enough and lesson 07's overfitting story begins...)")

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Passing 1-D `X` (a Series) to `.fit()` | sklearn demands `(n_samples, n_features)` | `df[["col"]]` or `.reshape(-1, 1)` |
| Reading R² as "percent correct" | It's variance explained vs a mean baseline — can even be negative | Report alongside RMSE/MAE in original units |
| Extrapolating far outside training range | The line happily predicts absurd prices beyond the data | Flag out-of-range inputs; retrain with wider data |
| Fitting a line through curved data | Biased predictions everywhere, visible as curved residuals | Check residual plots; add polynomial/log features |
| Comparing MSE across different target scales | lakh² vs dollar² numbers are incomparable | Standardise targets or compare R² |
| Ignoring correlated features (size & rooms) | Individual coefficients become unstable/misleading | Judge the *model*, not single weights; consider regularisation |

## 💡 Best Practices & Pro Tips

- **Start linear, always.** It trains instantly, explains itself to any stakeholder,
  and sets the bar fancier models must beat.
- **Look at residuals before believing coefficients** — the plot catches wrong
  shapes that R² hides.
- **Keep the split discipline** from lesson 03: fit on train, report test.
- **Transform skewed targets** (log-price is classic in real estate) — linear models
  love symmetric targets.
- **AI-engineering relevance:** linear regression inside a production pipeline is
  still the backbone of forecasting and pricing systems — cheap, auditable, and
  regulator-friendly where deep nets are neither.

## 📌 Summary

| Method | What it does | Example |
|---|---|---|
| `LinearRegression().fit(X, y)` | Least-squares fit of ŷ = w·x + b | Closed-form, no randomness |
| `model.coef_` / `model.intercept_` | Learned slope(s) / baseline | One coef per feature |
| `model.predict(X_new)` | Inference for new inputs | 2-D input required |
| `mean_absolute_error` | Mean absolute miss, target units | Robust headline metric |
| `mean_squared_error` (√ for RMSE) | Squared-error loss | Punishes big misses |
| `r2_score` | Variance explained vs mean baseline | 1.0 perfect, 0 = mean-guessing |
| `PolynomialFeatures(degree=d)` | Adds x², x³… columns | Curves via a linear engine |

Key takeaways:
- Training = choosing the line that minimises squared residuals; the result is
  fully described by `coef_` and `intercept_`.
- Quote RMSE/MAE for magnitude, R² for relative skill — never call R² accuracy.
- Residual plots are the cheapest, loudest diagnostic you own.
- Straight lines are a modelling choice; polynomials extend it without changing engines.

## 🔗 Next Lesson

Continue to **[05_Classification](../05_Classification/notes.ipynb)** — from
predicting amounts to predicting categories, where the metrics get subtle.